# 02 — Data Preprocessing

**CrimeMapAI | Primary modelling dataset: 2017–2022**

This notebook converts the raw districtwise cyber-crime data into a leakage-safe, district-year modelling panel.

### Frozen data decision
- Use the **2017–2022 dataset** for the primary ML pipeline.
- Exclude 2024 from primary training/target construction because its IPC/BNS-era classification changes are not source-verified against the historical indicators.
- The raw dataset's row grain is **registration-circle × district × year**, not district × year.
- For CrimeMapAI's district-level modelling objective, registration-circle rows are aggregated to **district × year** before temporal feature engineering.

### Primary prediction task
Predict the **next year's district-level `total_offences_under_ip`** from information available through the current year.

For a district-year `t`:
`target_next_year = total_offences_under_ip(t+1)`

The current-year target is retained only as a historical feature (e.g. lag-1), never as a same-row predictor of the next-year outcome.


In [31]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW_PATH = Path("../data/raw/districtwise-cyber-crimes-2017-onwards.csv")
assert RAW_PATH.exists(), f"Raw dataset not found: {RAW_PATH}"

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Loaded: {RAW_PATH}")
print(f"Shape: {df_raw.shape}")
df_raw.head()


Loaded: ..\data\raw\districtwise-cyber-crimes-2017-onwards.csv
Shape: (5322, 48)


,id,year,state_name,state_code,district_name,district_code,registration_circles,tampering_computer_source_documents,ransom_ware,offences_other_than_ransom_ware,...,stamps_counterfeiting,cyber_blackmailing_threatening,fake_news_on_social_media,other_offences,total_offences_under_ip,gambling_act,lotteries_act,copy_right_act,trade_marks_act,other_sll_crimes
0,0,2017,Jammu And Kashmir,1,Anantnag,1,Anantnag,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,2017,Jammu And Kashmir,1,Pulwama,11,Awantipora,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,2017,Jammu And Kashmir,1,Bandipora,623,Bandipora,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,2017,Jammu And Kashmir,1,Baramulla,3,Baramulla,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,2017,Jammu And Kashmir,1,Budgam,2,Budgam,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 1. Schema and integrity checks

In [32]:
META_COLUMNS = {
    "id", "year", "state_name", "state_code",
    "district_name", "district_code", "registration_circles"
}

required = META_COLUMNS
missing_required = required - set(df_raw.columns)
assert not missing_required, f"Missing required columns: {missing_required}"

assert df_raw["year"].between(2017, 2022).all()
assert df_raw.isna().sum().sum() == 0, "Unexpected missing values found."

numeric_crime_columns = [
    c for c in df_raw.columns
    if c not in META_COLUMNS and pd.api.types.is_numeric_dtype(df_raw[c])
]

negative_counts = (df_raw[numeric_crime_columns] < 0).sum().sum()
assert negative_counts == 0, f"Found {negative_counts} negative crime counts."

print("Missing values:", df_raw.isna().sum().sum())
print("Negative crime counts:", negative_counts)
print("Years:", sorted(df_raw["year"].unique()))
print("Raw rows:", len(df_raw))


Missing values: 0
Negative crime counts: 0
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Raw rows: 5322


## 2. Establish the actual data grain

A critical finding from the raw data is that `state_code + district_code + year` is **not unique**. Multiple registration circles can belong to the same district-year.

Therefore, treating each raw row as a district observation would overweight districts with more registration circles.

We aggregate crime counts across `registration_circles` to obtain the modelling grain:

**state × district × year**

The `id` column is an ingestion/row identifier and is discarded from modelling.


In [33]:
district_key = ["state_code", "district_code", "year"]

duplicate_district_keys = df_raw.duplicated(district_key).sum()
print("Rows participating in repeated district-year keys:", duplicate_district_keys)
print("Unique district-year keys:", df_raw[district_key].drop_duplicates().shape[0])

# Show the number of registration-circle rows per district-year.
grain_counts = (
    df_raw.groupby(district_key, dropna=False)
    .size()
    .rename("registration_circle_rows")
)

print(grain_counts.describe())
print("\nRegistration-circle row count distribution:")
print(grain_counts.value_counts().sort_index().head(15))


Rows participating in repeated district-year keys: 975
Unique district-year keys: 4347
count    4347.000000
mean        1.224293
std         0.732883
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        10.000000
Name: registration_circle_rows, dtype: float64

Registration-circle row count distribution:
registration_circle_rows
1     3751
2      399
3      107
4       61
5        3
6        8
7       11
9        2
10       5
Name: count, dtype: int64


In [34]:
crime_columns = [
    c for c in df_raw.columns
    if c not in META_COLUMNS
]

# Crime columns are all numeric in this dataset.
assert all(pd.api.types.is_numeric_dtype(df_raw[c]) for c in crime_columns)

agg_map = {c: "sum" for c in crime_columns}

# Geography labels are retained from the district.
district_meta = (
    df_raw.groupby(district_key, as_index=False)
    .agg(
        state_name=("state_name", "first"),
        district_name=("district_name", "first"),
    )
)

crime_agg = (
    df_raw.groupby(district_key, as_index=False)[crime_columns]
    .sum()
)

df_district = district_meta.merge(
    crime_agg,
    on=district_key,
    how="inner",
    validate="one_to_one",
)

assert not df_district[district_key].duplicated().any()
print("District-year panel:", df_district.shape)
df_district.head()


District-year panel: (4347, 46)


,state_code,district_code,year,state_name,district_name,tampering_computer_source_documents,ransom_ware,offences_other_than_ransom_ware,dishonestly_recv_stolen_cmp_resrc_or_comm_device,identity_theft,...,stamps_counterfeiting,cyber_blackmailing_threatening,fake_news_on_social_media,other_offences,total_offences_under_ip,gambling_act,lotteries_act,copy_right_act,trade_marks_act,other_sll_crimes
0,1,1,2017,Jammu And Kashmir,Anantnag,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,2018,Jammu And Kashmir,Anantnag,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,2019,Jammu And Kashmir,Anantnag,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,2020,Jammu And Kashmir,Anantnag,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,2021,Jammu And Kashmir,Anantnag,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Validate aggregation assumptions

The aggregation is a modelling transformation, not a claim that all source rows are independent district observations.

We verify:
- every district-year has exactly one aggregated row;
- all crime variables remain non-negative;
- no missing values were introduced;
- geography labels are internally consistent within each district code/year.


In [35]:
assert df_district[district_key].duplicated().sum() == 0
assert df_district[crime_columns].isna().sum().sum() == 0
assert (df_district[crime_columns] < 0).sum().sum() == 0

label_consistency = (
    df_raw.groupby(["state_code", "district_code", "year"])
    .agg(
        state_name_n=("state_name", "nunique"),
        district_name_n=("district_name", "nunique"),
    )
)

print("District-year groups with >1 state label:",
      (label_consistency["state_name_n"] > 1).sum())
print("District-year groups with >1 district label:",
      (label_consistency["district_name_n"] > 1).sum())


District-year groups with >1 state label: 12
District-year groups with >1 district label: 6


## 4. Target definition

The primary target is:

`total_offences_under_ip` in the **next year** for the same district.

We deliberately do not construct a target by summing every crime column. The dataset already supplies `total_offences_under_ip`, and the earlier audit showed that it is not simply the row-wise sum of all available indicators.

This makes the source-defined target preferable to inventing a new aggregate without a documented definition.


In [36]:
TARGET = "total_offences_under_ip"

target_by_year = df_district.groupby("year")[TARGET].agg(["count", "sum", "mean", "median", "max"])
target_by_year


,count,sum,mean,median,max
year,,,,,
2017,706,7976,11.297450,0.0,1336
2018,707,8647,12.230552,0.0,1450
2019,721,13730,19.042996,0.0,2508
2020,730,20201,27.672603,1.0,2388
2021,731,25384,34.725034,1.0,4379
2022,752,33786,44.928191,1.0,5395


## 5. Calendar-aware temporal feature engineering

Lag and target features are joined by **explicit calendar years** rather than positional `shift()`. This is important because a district may be absent in a particular year. A row from 2020 must not silently treat 2018 as its 2019 lag.

Rolling statistics are based only on the immediately preceding two calendar years. Missing years remain missing rather than being bridged.


In [37]:
group_cols = ["state_code", "district_code"]
feature_base = crime_columns.copy()

# Start from the validated district-year panel.
df_panel = df_district.sort_values(group_cols + ["year"]).reset_index(drop=True)

# Calendar-safe lag construction.
# A row at year t receives lag-1 only from the same district at t-1,
# and lag-2 only from the same district at t-2. Missing years are not bridged.
prev1 = df_district[group_cols + ["year"] + feature_base].copy()
prev1["year"] += 1
prev1 = prev1.rename(columns={c: f"{c}_lag1" for c in feature_base})

prev2 = df_district[group_cols + ["year"] + feature_base].copy()
prev2["year"] += 2
prev2 = prev2.rename(columns={c: f"{c}_lag2" for c in feature_base})

df_panel = df_panel.merge(
    prev1, on=group_cols + ["year"], how="left", validate="one_to_one"
)
df_panel = df_panel.merge(
    prev2, on=group_cols + ["year"], how="left", validate="one_to_one"
)

# Two-year historical summaries. Both columns refer strictly to prior calendar years.
for c in feature_base:
    lag1 = df_panel[f"{c}_lag1"]
    lag2 = df_panel[f"{c}_lag2"]
    vals = pd.concat([lag2.rename("lag2"), lag1.rename("lag1")], axis=1)
    df_panel[f"{c}_rolling_mean2"] = vals.mean(axis=1, skipna=True)
    df_panel[f"{c}_rolling_std2"] = vals.std(axis=1, skipna=False)

# Target-history features.
df_panel["target_lag1"] = df_panel[f"{TARGET}_lag1"]
df_panel["target_lag2"] = df_panel[f"{TARGET}_lag2"]
df_panel["target_yoy_change"] = df_panel[TARGET] - df_panel["target_lag1"]
df_panel["target_yoy_growth"] = (
    df_panel["target_yoy_change"] /
    df_panel["target_lag1"].replace(0, np.nan)
)

# Exact next-calendar-year target.
next_target = df_district[group_cols + ["year", TARGET]].copy()
next_target["year"] -= 1
next_target = next_target.rename(columns={TARGET: "target_next_year"})
df_panel = df_panel.merge(
    next_target, on=group_cols + ["year"], how="left", validate="one_to_one"
)

print("Rows before dropping unavailable target:", len(df_panel))
print("Rows with next-calendar-year target:", df_panel["target_next_year"].notna().sum())


Rows before dropping unavailable target: 4347
Rows with next-calendar-year target: 3594


## 6. Remove rows that cannot support next-year prediction

The final supervised-learning table contains only district-years with an observed next year.

This naturally removes 2022 from the primary target table because 2023 is absent from the supplied 2017–2022 dataset.

We do **not** fabricate 2023 or interpolate a target.


In [38]:
df_model = df_panel[df_panel["target_next_year"].notna()].copy()

print("Model rows:", len(df_model))
print("Model years:", sorted(df_model["year"].unique()))
print("Rows by prediction origin year:")
print(df_model["year"].value_counts().sort_index())


Model rows: 3594
Model years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Rows by prediction origin year:
year
2017    706
2018    707
2019    721
2020    730
2021    730
Name: count, dtype: int64


## 7. Missingness created by lag features

Lag-based features are expected to be unavailable at the beginning of a district's history.

We retain the rows and let the modelling pipeline handle feature missingness rather than silently imputing historical values here.

The final estimator stage will use a scikit-learn pipeline so imputation, scaling/encoding, and model fitting are learned from the training folds only.


In [39]:
lag_columns = [c for c in df_model.columns if "_lag" in c or "_rolling_" in c or c.endswith("_yoy_growth")]
missing_summary = (
    df_model[lag_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
)
missing_summary


target_yoy_growth                                                0.634947
total_offences_under_ip_lag2                                     0.406511
gambling_act_lag2                                                0.406511
fake_news_on_social_media_lag2                                   0.406511
cyber_blackmailing_threatening_lag2                              0.406511
pub_or_trans_mtrl_cont_sxly_explct_act_elect_frm_rolling_std2    0.406511
other_offences_lag2                                              0.406511
un_athryz_access_atmpt_access_prct_comp_sys_lag2                 0.406511
stamps_counterfeiting_lag2                                       0.406511
fake_profile_lag2                                                0.406511
defamation_morphing_lag2                                         0.406511
other_frauds_lag2                                                0.406511
otp_frauds_lag2                                                  0.406511
forgery_lag2                          

## 8. Temporal modelling split

The modelling table contains only rows for which the **immediately following calendar year** is observed. The final observed year in the source therefore cannot be used as an origin for next-year prediction.


In [40]:
# Save processed artifacts for subsequent notebooks.
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

panel_path = OUTPUT_DIR / "district_year_panel.csv"
model_path = OUTPUT_DIR / "model_table.csv"

df_district.to_csv(panel_path, index=False)
df_model.to_csv(model_path, index=False)

print("Saved:")
print(panel_path)
print(model_path)


Saved:
..\data\processed\district_year_panel.csv
..\data\processed\model_table.csv


## 9. Preprocessing audit summary

Before proceeding to EDA/modeling, verify these statements:

- [x] 2024 excluded from the primary modelling dataset.
- [x] `id` excluded as a modelling key.
- [x] Raw registration-circle rows aggregated to district-year.
- [x] No random splitting.
- [x] Next-year target generated with a within-district temporal shift.
- [x] No artificial 2023 target created.
- [x] Source-defined `total_offences_under_ip` retained rather than replaced by an invented sum.
- [x] Lag/rolling features are generated using past information only.
- [x] Processed artifacts are written under `data/processed/`.

**Next notebook:** `03_eda.ipynb`
